### Setup & Prerequisites

To execute this notebook, set up the environment using terminal commands:

1. Create the Conda environment from the configuration file:
   ```bash
   conda env create -f environment.yml
   ```
2. Activate the environment:
   ```bash
   conda activate isles2026
   ```
3. Select Kernel:
   Make sure the active Jupyter kernel is set to isles2026 before running the cells below.
---

<FollowUp label="Want me to generate a complete README.md template for your project?" query="Create a clean, professional README.md template for an ISLES 2026 medical imaging challenge project including setup, structure, and usage sections."/>

In [ ]:
import os
import shutil
import json
from pathlib import Path


In [ ]:

# --- CONFIGURATION ---
# Using the exact path and raw string format (r"...") for Windows
ATLAS_SOURCE_DIR = r"C:\Users\Tevel Katzir\Downloads\ATLAS_R2.1_raw" 
OUTPUT_DIR = r"C:\Users\Tevel Katzir\Desktop\Dataset101_ATLAS"
NUM_SUBJECTS = 5 
# ---------------------

print("Creating nnU-Net directories...")
imagesTr_dir = os.path.join(OUTPUT_DIR, "imagesTr")
labelsTr_dir = os.path.join(OUTPUT_DIR, "labelsTr")
os.makedirs(imagesTr_dir, exist_ok=True)
os.makedirs(labelsTr_dir, exist_ok=True)

print(f"Searching for T1w MRI scans in {ATLAS_SOURCE_DIR}...")
# Be very specific to match the ATLAS R2.1 naming you provided
t1w_files = list(Path(ATLAS_SOURCE_DIR).rglob("*_desc-brain_T1w.nii.gz"))

if len(t1w_files) == 0:
    print("🚨 ERROR: Still couldn't find the files. Check the ATLAS_SOURCE_DIR path!")
else:
    count = 0
    for t1_path in t1w_files:
        if count >= NUM_SUBJECTS:
            break
            
        # Exact replacement string based on the files you provided
        mask_name = t1_path.name.replace("_desc-brain_T1w.nii.gz", "_label-lesion_desc-T1lesion_mask.nii.gz")
        mask_path = t1_path.parent / mask_name
        
        if mask_path.exists():
            subject_id = f"ATLAS_{count+1:03d}"
            
            nnunet_t1_name = f"{subject_id}_0000.nii.gz"
            nnunet_mask_name = f"{subject_id}.nii.gz"
            
            print(f"Copying {t1_path.name} \n  -> {nnunet_t1_name}...")
            shutil.copy(t1_path, os.path.join(imagesTr_dir, nnunet_t1_name))
            shutil.copy(mask_path, os.path.join(labelsTr_dir, nnunet_mask_name))
            
            count += 1
        else:
            print(f"⚠️ Warning: Found T1 scan but missing mask for {t1_path.name}")

    # Create dataset.json
    print("\nGenerating dataset.json...")
    dataset_json = {
        "channel_names": {
            "0": "T1"
        },
        "labels": {
            "background": 0,
            "lesion": 1
        },
        "numTraining": count,
        "file_ending": ".nii.gz"
    }

    with open(os.path.join(OUTPUT_DIR, "dataset.json"), "w") as f:
        json.dump(dataset_json, f, indent=4)

    print(f"\n✅ Success! Built a tiny dataset with {count} subjects at {OUTPUT_DIR}.")

In [ ]:
import subprocess

# --- 1. SET UP THE WORKSPACE ---
WORKSPACE_DIR = r"C:\Users\Tevel Katzir\Desktop\nnUNet_workspace"
RAW_DIR = os.path.join(WORKSPACE_DIR, "nnUNet_raw")
PREPROCESSED_DIR = os.path.join(WORKSPACE_DIR, "nnUNet_preprocessed")
RESULTS_DIR = os.path.join(WORKSPACE_DIR, "nnUNet_results")

os.makedirs(PREPROCESSED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

# Move your dataset inside nnUNet_raw so the framework can find it
old_dataset_path = r"C:\Users\Tevel Katzir\Desktop\Dataset101_ATLAS"
new_dataset_path = os.path.join(RAW_DIR, "Dataset101_ATLAS")

if os.path.exists(old_dataset_path):
    shutil.move(old_dataset_path, new_dataset_path)
    print("✅ Moved Dataset101_ATLAS into the workspace.")

# --- 2. SET THE REQUIRED ENVIRONMENT VARIABLES ---
# This forces nnU-Net to read/write to your Desktop workspace instead of system defaults
os.environ["nnUNet_raw"] = RAW_DIR
os.environ["nnUNet_preprocessed"] = PREPROCESSED_DIR
os.environ["nnUNet_results"] = RESULTS_DIR

# --- 3. EXECUTE THE PIPELINE COMMANDS ---
print("\n🚀 [Step 1/2] Fingerprinting and Preprocessing...")
# shell=True ensures Windows finds the pip-installed commands correctly
subprocess.run("nnUNetv2_plan_and_preprocess -d 101 --verify_dataset_integrity", shell=True, check=True)

print("\n🚀 [Step 2/2] Starting Training...")
print("⚠️ Note: You are training a 2D model on your CPU (-device cpu).")
print("⚠️ It will be very slow. Press Ctrl+C in your terminal to cancel it once you've seen it start successfully.")

# We use '2d' because '3d' requires significantly more memory, and we add '-device cpu' to bypass the GPU check
subprocess.run("nnUNetv2_train 101 2d 0 -device cpu", shell=True, check=True)